In [ ]:
import re, os, random
import numpy as np
import pandas as pd
import math
from tqdm import tqdm
import string
import time
import pickle
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from collections import Counter

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
AA_VOCAB  = list('ACDEFGHIKLMNPQRSTVWXY')
SS8_VOCAB = ['C', 'B', 'E', 'G', 'I', 'H', 'S', 'T']
NUM_CLASSES = len(SS8_VOCAB)
MAX_LEN = 700
REPLACEMENT_DICT = {
    ("A","V"), ("S","T"), ("F","Y"), ("K","R"), ("C","M"), ("D","E"), ("N","Q"), ("L","I"),
    ("V","A"), ("T","S"), ("Y","F"), ("R","K"), ("M","C"), ("E","D"), ("Q","N"), ("I","L")
} # no replacement for G, H, P, W, X

class ProteinDataset(Dataset):
    def __init__(self, df, p, aug_method, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.max_len = max_len
        self.aa_map = {c: i for i, c in enumerate(AA_VOCAB)}
        self.ss_map = {c: i for i, c in enumerate(SS8_VOCAB)}
        self.p = p
        self.aug_method = aug_method # 'dictionary', 'alaine' or None

        self.lookup = np.arange(256, dtype=np.uint8)
        for src, dst in REPLACEMENT_DICT:
            self.lookup[ord(src)] = ord(dst)

    def __len__(self):
        return len(self.df)

    def encode(self, seq, vocab_map):
        arr = np.zeros(self.max_len, dtype=np.int64)
        for i, ch in enumerate(seq[:self.max_len]):
            if ch in vocab_map:
                arr[i] = vocab_map[ch]
        return arr

    def replace_dictionary(self, sequence):
        mask = np.random.random(len(sequence)) < self.p
        replaced = self.lookup[sequence]
        return np.where(mask, replaced, sequence)

    def replace_alanine(self, sequence):
        mask = np.random.random(len(sequence)) < self.p
        alanine = np.full_like(sequence, ord('A'))
        return np.where(mask, alanine, sequence)

    def augment(self, seq_str):
        seq = np.frombuffer(seq_str.encode(), dtype=np.uint8).copy()
        if self.aug_method == 'dictionary':
            seq = self.replace_dictionary(seq)
        elif self.aug_method == 'alanine':
            seq = self.replace_alanine(seq)
        return ''.join(chr(c) for c in seq)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = self.augment(row['input']) if self.aug_method is not None else row['input']  # apply augmentation before encoding
        X = self.encode(seq, self.aa_map)
        y = self.encode(row['dssp8'], self.ss_map)
        length = min(len(row['input']), self.max_len)
        mask = np.zeros(self.max_len, dtype=np.float32)
        mask[:length] = 1.0
        return (torch.tensor(X),
                torch.tensor(y),
                torch.tensor(mask))